# DS Automation Assignment

# Task 1
Using our prepared churn data from week 2:
- use pycaret to find an ML algorithm that performs best on the data
    - Choose a metric you think is best to use for finding the best model; by default, it is accuracy but it could be AUC, precision, recall, etc. The week 3 FTE has some information on these different metrics.
- save the model to disk
- create a Python script/file/module with a function that takes a pandas dataframe as an input and returns the probability of churn for each row in the dataframe
    - your Python file/function should print out the predictions for new data (new_churn_data.csv)
    - the true values for the new data are [1, 0, 0, 1, 0] if you're interested
- test your Python module and function with the new data, new_churn_data.csv
- write a short summary of the process and results at the end of this notebook
- upload this Jupyter Notebook and Python file to a Github repository, and turn in a link to the repository in the week 5 assignment dropbox

Additional challenges:
- return the probability of churn for each new prediction, and the percentile where that prediction is in the distribution of probability predictions from the training dataset (e.g. a high probability of churn like 0.78 might be at the 90th percentile)
- use other autoML packages, such as TPOT, H2O, MLBox, etc, and compare performance and features with pycaret
- create a class in your Python module to hold the functions that you created
- accept user input to specify a file using a tool such as Python's `input()` function, the `click` package for command-line arguments, or a GUI
- Use the unmodified churn data (new_unmodified_churn_data.csv) in your Python script. This will require adding the same preprocessing steps from week 2 since this data is like the original unmodified dataset from week 1.


# Task 2: Critical Reflection
What are your observations based on the analyses you had performed? Is there one model that seems particularly useful for predicting churn. Why or why not?

In [30]:
#    build dependencies
from pycaret.classification import setup, compare_models, predict_model, save_model, load_model
import sklearn
from sklearn.naive_bayes import GaussianNB
import pandas as pd
import pickle
from IPython.display import Code


## Use pycaret to find an ML algorithm that performs best on the data
    - Choose a metric you think is best to use for finding the best model; by default, it is accuracy but it could be AUC, precision, recall, etc. The week 3 FTE has some information on these different metrics.

In [31]:
train_data = pd.read_csv('Data/prepared_churn_data.csv', index_col='customerID')
train_data = train_data.drop(columns='AvgMonthly')

#    low importance features identified in the Week 4 assignment
low_importance_features = ['DeviceProtection', 'PaperlessBilling', 'StreamingTV', 'StreamingMovies', 'MultipleLines', 'gender', 'Partner', 'SeniorCitizen', 'Dependents', 'PhoneService']

automl = setup(train_data, target='Churn', ignore_features=low_importance_features, preprocess='true', remove_outliers='true')


,Description,Value
0,Session id,1238
1,Target,Churn
2,Target type,Binary
3,Original data shape,"(7043, 7)"
4,Transformed data shape,"(6796, 6)"
5,Transformed train set shape,"(4683, 6)"
6,Transformed test set shape,"(2113, 6)"
7,Ignore features,10
8,Numeric features,5
9,Preprocess,true


Since only one of the low importance features is part of the training data only one was dropped.  With remove outliers set to true rows were dropped in the data set, train set sizes.  There were 247 rows were removed from the transformed data and train set sizes.  

In [33]:
best_model = compare_models(sort='Kappa')

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.7909,0.8203,0.5191,0.6295,0.5677,0.4317,0.4360,0.6610
gbc,Gradient Boosting Classifier,0.7941,0.8317,0.5015,0.6452,0.5628,0.4313,0.4380,0.0350
ada,Ada Boost Classifier,0.7899,0.8298,0.5122,0.6279,0.5633,0.4270,0.4314,0.0170
lr,Logistic Regression,0.7923,0.8280,0.4946,0.6428,0.5575,0.4251,0.4322,0.0130
lda,Linear Discriminant Analysis,0.7905,0.8169,0.4947,0.6385,0.5558,0.4218,0.4287,0.0120
ridge,Ridge Classifier,0.7923,0.8169,0.4595,0.6566,0.5392,0.4108,0.4224,0.0110
qda,Quadratic Discriminant Analysis,0.7209,0.8170,0.7722,0.4842,0.5948,0.3987,0.4244,0.0110
nb,Naive Bayes,0.7197,0.8097,0.7661,0.4824,0.5917,0.3947,0.4196,0.0110
rf,Random Forest Classifier,0.7779,0.7933,0.4869,0.6009,0.5367,0.3931,0.3975,0.0320
et,Extra Trees Classifier,0.7657,0.7690,0.4778,0.5717,0.5200,0.3668,0.3697,0.0270


An interesting note; before restarting the environment the Ridge Classifier and the Gradient Boosting Classifer matched in accuracy and the GBC model had the most categories of highest rating.  Now i am not receiving consistant results on the highest rated models.

In [34]:
print("Training data shape:    ",train_data.iloc[-2:-1].shape,"\n")
predict_model(best_model, train_data.iloc[-2:-1])

Training data shape:     (1, 7) 



,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,1.0000,0,1.0000,1.0000,1.0000,nan,0.0000


,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,Churn,prediction_label,prediction_score
customerID,,,,,,,,,
8361-LTMKD,4,1,0,1,74.400002,306.600006,1,1,0.6463


In [35]:
best_model

LGBMClassifier(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
               importance_type='split', learning_rate=0.1, max_depth=-1,
               min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
               n_estimators=100, n_jobs=-1, num_leaves=31, objective=None,
               random_state=1238, reg_alpha=0.0, reg_lambda=0.0, subsample=1.0,
               subsample_for_bin=200000, subsample_freq=0)

## Save the model to disk

In [36]:
save_model(best_model, 'Models/LBGM')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['tenure', 'Contract',
                                              'PaymentMethod', 'MonthlyCharges',
                                              'TotalCharges'],
                                     transformer=SimpleImputer(add_indicator=False,
                                                               copy=True,
                                                               fill_value=None,
                                                               keep_empty_features=False,
                                                               missing_values=nan,
                                                               strategy='mean'))),
                 ('categorical_imputer',
                  TransformerWrap...
                  LGBMClassifier(boosting_type='gbdt', class_weight=None,
            

## Create a Python script/file/module with a function that takes a pandas dataframe as an input and returns the probability of churn for each row in the dataframe
    - your Python file/function should print out the predictions for new data (new_churn_data.csv)
    - the true values for the new data are [1, 0, 0, 1, 0] if you're interested

## test your Python module and function with the new data, new_churn_data.csv

In [37]:
Code('predictChurn_v2.py')

import sys
import pandas as pd
from pycaret.classification import predict_model, load_model

class ChurnPredictor:
    def __init__(self, model_path, train_path=None):
        self.model = load_model(model_path)
        self.train_probability = None
        if train_path:
            train_df = self.load_data(train_path)
            train_df = train_df.drop(columns=['Churn', 'AvgMonthly'], errors='ignore')
            self.train_probability = self.get_churn_probability(train_df)

    def predict(self, df):
        predictions = predict_model(self.model, data=df)

        predictions.rename(
            {'prediction_label': 'Churn_prediction'},
            axis=1,
            inplace=True
        )

        predictions['Churn_prediction'].replace(
            {1: 'Churn', 0: 'No churn'},
            inplace=True
        )

        return predictions['Churn_prediction']
    
    def score_predict(self, df):
        predictions = predict_model(self.model, data=df)

        predictions.rename(
            {'prediction_label': 'Churn_prediction', 'prediction_score': 'Churn_probability'},
            axis=1,
            inplace=True
        )

        predictions['Churn_prediction'] = predictions['Churn_prediction'].replace(
            {1: 'Churn', 0: 'No churn'}
        )

        return predictions[['Churn_prediction', 'Churn_probability']]


def load_data(filepath):
    return pd.read_csv(filepath, index_col='customerID')


if __name__ == "__main__":
    if len(sys.argv) > 3:
        print("Usage: python predictChurn.py <data_file> <model_path>")
        sys.exit(1)

    data_file = sys.argv[1]
    model_path = sys.argv[2]

    #df = load_data('Data/new_churn_data.csv')
    df = load_data(data_file)

    predictor = ChurnPredictor(model_path)

    if "Ridge" in model_path or "SVM" in model_path:
        predictions = predictor.predict(df)
    else:
        predictions = predictor.score_predict(df)

    print("predictions:")
    print(predictions)

In [38]:
%run predictChurn_v2.py Data/new_churn_data.csv Models/LBGM


Transformation Pipeline and Model Successfully Loaded


predictions:
           Churn_prediction  Churn_probability
customerID                                    
9305-CKSKC         No churn             0.5067
1452-KNGVK         No churn             0.7623
6723-OKKJM         No churn             0.8165
7832-POPKP         No churn             0.8795
6348-TACGU         No churn             0.9078


## Upload this Jupyter Notebook and Python file to a Github repository, and turn in a link to the repository in the week 5 assignment dropbox

# Additional challenges:

## return the probability of churn for each new prediction, and the percentile where that prediction is in the distribution of probability predictions from the training dataset (e.g. a high probability of churn like 0.78 might be at the 90th percentile)

## Create a class in your Python module to hold the functions that you created
## Accept user input to specify a file using a tool such as Python's `input()` function, the `click` package for command-line arguments, or a GUI
## Use the unmodified churn data (new_unmodified_churn_data.csv) in your Python script. This will require adding the same preprocessing steps from week 2 since this data is like the original unmodified dataset from week 1.

In [39]:
new_data = pd.read_csv('Data/new_churn_data_unmodified.csv')
print(new_data.isna().sum())

customerID        0
tenure            0
PhoneService      0
Contract          0
PaymentMethod     0
MonthlyCharges    0
TotalCharges      0
dtype: int64


In [40]:
new_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customerID      5 non-null      object 
 1   tenure          5 non-null      int64  
 2   PhoneService    5 non-null      object 
 3   Contract        5 non-null      object 
 4   PaymentMethod   5 non-null      object 
 5   MonthlyCharges  5 non-null      float64
 6   TotalCharges    5 non-null      float64
dtypes: float64(2), int64(1), object(4)
memory usage: 408.0+ bytes


In [41]:
new_data['PhoneService'] = new_data['PhoneService'].map({'No': 0, 'Yes': 1})
new_data['Contract'] = new_data['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
new_data['PaymentMethod'] = new_data['PaymentMethod'].map({'Electronic check': 0, 'Mailed check': 1, 'Bank transfer (automatic)': 2, 'Credit card (automatic)': 3})

new_data.to_csv('Data/new_churn_data_cleaned.csv')

In [43]:
print('Ridge Classifier:')
%run predictChurn_v2.py Data/new_churn_data_cleaned.csv Models/Ridge
print('\nGradient Boosting Classifier:')
%run predictChurn_v2.py Data/new_churn_data_cleaned.csv Models/GBC
print('\nAda Boost Classifier:')
%run predictChurn_v2.py Data/new_churn_data_cleaned.csv Models/ADA
print('\nLogistic Regression:')
%run predictChurn_v2.py Data/new_churn_data_cleaned.csv Models/LR
print('\nNaive Bayes:')
%run predictChurn_v2.py Data/new_churn_data_cleaned.csv Models/NB
print('\nQuadratic Discriminant Analysis:')
%run predictChurn_v2.py Data/new_churn_data_cleaned.csv Models/QDA
print('\nLight Gradient Boosting Machine:')
%run predictChurn_v2.py Data/new_churn_data_cleaned.csv Models/LBGM

Ridge Classifier:
Transformation Pipeline and Model Successfully Loaded


predictions:
customerID
9305-CKSKC       Churn
1452-KNGVK    No churn
6723-OKKJM    No churn
7832-POPKP       Churn
6348-TACGU    No churn
Name: Churn_prediction, dtype: object

Gradient Boosting Classifier:
Transformation Pipeline and Model Successfully Loaded


predictions:
           Churn_prediction  Churn_probability
customerID                                    
9305-CKSKC            Churn             0.6965
1452-KNGVK         No churn             0.8840
6723-OKKJM         No churn             0.8609
7832-POPKP         No churn             0.5429
6348-TACGU         No churn             0.9489

Ada Boost Classifier:
Transformation Pipeline and Model Successfully Loaded


predictions:
           Churn_prediction  Churn_probability
customerID                                    
9305-CKSKC            Churn             0.5021
1452-KNGVK         No churn             0.5087
6723-OKKJM         No churn             0.5100
7832-POPKP         No churn             0.5017
6348-TACGU         No churn             0.5157

Logistic Regression:
Transformation Pipeline and Model Successfully Loaded


predictions:
           Churn_prediction  Churn_probability
customerID                                    
9305-CKSKC            Churn             0.5572
1452-KNGVK         No churn             0.5788
6723-OKKJM         No churn             0.9349
7832-POPKP         No churn             0.8079
6348-TACGU         No churn             0.8776

Naive Bayes:
Transformation Pipeline and Model Successfully Loaded


predictions:
           Churn_prediction  Churn_probability
customerID                                    
9305-CKSKC            Churn             0.9030
1452-KNGVK         No churn             0.7259
6723-OKKJM         No churn             0.7927
7832-POPKP         No churn             0.6954
6348-TACGU         No churn             1.0000

Quadratic Discriminant Analysis:
Transformation Pipeline and Model Successfully Loaded


predictions:
           Churn_prediction  Churn_probability
customerID                                    
9305-CKSKC            Churn             0.8357
1452-KNGVK         No churn             0.9747
6723-OKKJM         No churn             0.9979
7832-POPKP         No churn             1.0000
6348-TACGU         No churn             1.0000

Light Gradient Boosting Machine:
Transformation Pipeline and Model Successfully Loaded


predictions:
           Churn_prediction  Churn_probability
customerID                                    
9305-CKSKC            Churn             0.5212
1452-KNGVK         No churn             0.7623
6723-OKKJM            Churn             0.5077
7832-POPKP         No churn             0.8697
6348-TACGU         No churn             0.9955


# Task 2: Critical Reflection
What are your observations based on the analyses you had performed? Is there one model that seems particularly useful for predicting churn. Why or why not?

The model comparison has been inconsistent after restarting the environment.  The Ridge Classifier seems to be the most consistent in the comparison for Accuracy and Precision.  The Gradient Boosting Classifier frequently shows the most categories with the highest rating.  I would lean toward using the Ridge Classifier model since it has been the most consistent in the model comparison.  I would probably use the Gradient Boosting Classifier model if a probability is desired because it has been the most consistantly highly rated by the comparison.

# Summary
## Including summary of the process and results

In the training data low importance features identified in the week 4 assignment were dropped along with the AvgMonthly feature that was an added feature from week 2.  This data was used to identify a model to then use to predict churn on new data sets.  The model was saved then used in a python script predict churn from the new data sets.  In the additional challenges the new_churn_data_unmodified.csv data was checked for missing data then the non-numeric columns were converted to numeric.  The cleaned data was saved then run through the script to predict churn for the customers included in that data set.

I kept getting errors when running the Ridge Classifier model through the script that was adding a probability score.  Google informed me that the RC model does not support scoring.  I asked Google what other models don't support scoring and based on that i copied the original function for predicting from the v1 script.  If the model selected is in the list i call the original prediction otherwise i call the new function that includes probability scoring.

# AI Disclosure

Used Google/Gemini for package methods, method parameters, and syntax.

# Export to HTML

In [44]:
!jupyter nbconvert --to html "Russell_Josh_Assignment5.ipynb"

[NbConvertApp] Converting notebook Russell_Josh_Assignment5.ipynb to html
[NbConvertApp] Writing 373873 bytes to Russell_Josh_Assignment5.html
